# RAGAS Benchmark — Sherlock Holmes Corpus

This notebook runs the **RAGAS quality evaluation** on RAG methods using the Sherlock Holmes corpus.

**Dataset**: Adventures of Sherlock Holmes (full text) chunked into overlapping passages.
Test questions are generated via `scripts/generate_testset.py` and evaluated on 4 RAGAS metrics:
- **Faithfulness** — Is the answer grounded in the retrieved context?
- **Answer Relevancy** — Does the answer address the question?
- **Context Precision** — Are the retrieved chunks relevant to the question?
- **Context Recall** — Do the retrieved chunks cover the ground truth?

In [2]:
# --- Environment setup ---
from helpers.config import setup_environment, load_config
from helpers.types import BenchmarkConfig

# --- Configuration ---
# Edit these variables before running

METHODS = ["vdb"]                  # Adapter methods to benchmark
MAX_QUESTIONS = 10                  # Number of test questions (0 = all)
TOP_K = 8                           # Chunks to retrieve per query
CHUNK_SIZE = 1200                   # Characters per chunk
CHUNK_OVERLAP = 200                 # Overlap between chunks
SKIP_INDEX = False                  # Set True to reuse existing indexes

setup_environment()


config_dict = load_config(overrides={
    "methods": METHODS,
    "max_questions": MAX_QUESTIONS,
    "top_k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
})
config = BenchmarkConfig.from_dict(config_dict)
print(f"Config: {len(METHODS)} methods, {MAX_QUESTIONS} questions, top_k={TOP_K}")

ModuleNotFoundError: No module named 'helpers.types'

In [ ]:
# --- Load Dataset ---
from helpers.datasets import load_ragas_sherlock

chunks, testset = load_ragas_sherlock(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    max_questions=MAX_QUESTIONS,
)
print(f"\nSample question: {testset[0]['question']}")
print(f"Ground truth: {testset[0]['ground_truth'][:200]}...")

In [ ]:
# --- Run Benchmarks ---
from helpers.pipeline import run_all_methods

all_results = await run_all_methods(
    methods=METHODS,
    chunks=chunks,
    testset=testset,
    config=config,
    skip_index=SKIP_INDEX,
)

In [ ]:
# --- Evaluate with RAGAS ---
from helpers.metrics import evaluate_all_methods

all_results = await evaluate_all_methods(
    all_results,
    config=config,
    use_ragas=True,
    use_em_f1=False,
)

In [ ]:
# --- Summary Table ---
from helpers.charts import summary_table
summary_table(all_results)

In [ ]:
# --- Charts ---
from helpers.charts import ragas_bar_chart, latency_chart, cost_breakdown_chart

ragas_bar_chart(all_results, title="RAGAS Scores — Sherlock Holmes")
latency_chart(all_results, title="Avg Query Latency — Sherlock Holmes")
cost_breakdown_chart(all_results, title="Cost Breakdown — Sherlock Holmes")

In [ ]:
# --- Save Results ---
import json, os
from helpers.config import PROJECT_ROOT
from datetime import datetime, timezone

output_dir = PROJECT_ROOT / "results" / "notebook_ragas"
os.makedirs(output_dir, exist_ok=True)

for method_name, method_data in all_results.items():
    filepath = output_dir / f"{method_name}.json"
    payload = {
        "run_config": {
            "corpus": "sherlock_holmes",
            "num_questions": len(testset),
            "top_k": TOP_K,
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        },
        "method": method_name,
        **method_data,
    }
    with open(filepath, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"Saved: {filepath}")